# 06 Spark liest Kafka-Wetterdaten

Dieses Notebook implementiert BD-15. Es liest das Kafka Topic `weather.observations` mit Spark, parst die JSON-Messages in strukturierte Spalten, dedupliziert die Daten ueber `match_id` und speichert das Ergebnis als Parquet-Snapshot.

Output:

- `data/silver/weather_from_kafka.parquet`
- `outputs/tables/kafka_weather_read_status.csv`

## Methodischer Ansatz

Kafka wird als Transport-Schritt zwischen Wetter-Ingestion und Spark-Verarbeitung verwendet. Der Spark-Read erzeugt einen reproduzierbaren Batch-Snapshot aus dem Topic: alle aktuell vorhandenen Messages werden gelesen, mit einem festen Schema geparst und als Silver-Datenprodukt gespeichert.

Mehrfaches Ausfuehren des Producers kann dieselbe Wetterbeobachtung mehrfach ins Topic schreiben. Deshalb wird im Read-Schritt ueber `match_id` dedupliziert. Der Kafka-Key und das JSON-Feld `match_id` muessen dabei uebereinstimmen.

In [ ]:
import os

from project_paths import ensure_pyspark_path, project_root as get_project_root
from pipeline_config import (
    DEFAULT_KAFKA_BOOTSTRAP_SERVERS,
    DEFAULT_KAFKA_WEATHER_TOPIC,
    DEFAULT_SPARK_MASTER,
    WEATHER_SCHEMA_VERSION,
    expected_match_count,
)

ensure_pyspark_path()

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql import types as T

KAFKA_BOOTSTRAP_SERVERS = os.getenv('KAFKA_BOOTSTRAP_SERVERS', DEFAULT_KAFKA_BOOTSTRAP_SERVERS)
KAFKA_TOPIC = os.getenv('KAFKA_WEATHER_TOPIC', DEFAULT_KAFKA_WEATHER_TOPIC)
SPARK_MASTER = os.getenv('SPARK_MASTER', DEFAULT_SPARK_MASTER)
SPARK_KAFKA_PACKAGE = os.getenv(
    'SPARK_KAFKA_PACKAGE',
    'org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0',
)

{
    'spark_master': SPARK_MASTER,
    'kafka_bootstrap_servers': KAFKA_BOOTSTRAP_SERVERS,
    'kafka_topic': KAFKA_TOPIC,
    'spark_kafka_package': SPARK_KAFKA_PACKAGE,
}


## SparkSession erstellen

Der Kafka-Connector wird als Spark-Package konfiguriert. In Umgebungen, in denen der Connector bereits vorinstalliert ist, kann `SPARK_KAFKA_PACKAGE` leer gesetzt werden.

In [ ]:
builder = (
    SparkSession.builder
    .appName('football-weather-bd15-kafka-read')
    .master(SPARK_MASTER)
    .config('spark.sql.shuffle.partitions', '4')
)

if SPARK_KAFKA_PACKAGE:
    builder = builder.config('spark.jars.packages', SPARK_KAFKA_PACKAGE)

spark = builder.getOrCreate()
spark.sparkContext.setLogLevel('WARN')

spark.version

## Kafka Topic als Snapshot lesen

`startingOffsets = earliest` und `endingOffsets = latest` lesen einen festen Stand des Topics. Damit ist der Notebook-Lauf reproduzierbar und endet nach dem Speichern des Parquet-Ergebnisses.

In [ ]:
kafka_raw = (
    spark.read
    .format('kafka')
    .option('kafka.bootstrap.servers', KAFKA_BOOTSTRAP_SERVERS)
    .option('subscribe', KAFKA_TOPIC)
    .option('startingOffsets', 'earliest')
    .option('endingOffsets', 'latest')
    .load()
)

kafka_strings = kafka_raw.select(
    F.col('topic'),
    F.col('partition'),
    F.col('offset'),
    F.col('timestamp').alias('kafka_timestamp'),
    F.col('key').cast('string').alias('message_key'),
    F.col('value').cast('string').alias('message_value'),
)

raw_message_count = kafka_strings.count()
assert raw_message_count > 0, f'Kafka topic {KAFKA_TOPIC} contains no messages.'

kafka_strings.orderBy('offset').show(3, truncate=100)
raw_message_count

## JSON-Value mit Schema parsen

Das Schema entspricht dem Producer-Vertrag aus `docs/kafka_topics.md`. Fehlerhafte JSON-Messages wuerden als Zeilen mit leerem `data` sichtbar.

In [ ]:
weather_schema = T.StructType([
    T.StructField('match_id', T.LongType(), False),
    T.StructField('match_date', T.StringType(), False),
    T.StructField('kick_off', T.StringType(), False),
    T.StructField('kickoff_timestamp', T.StringType(), False),
    T.StructField('weather_time', T.StringType(), False),
    T.StructField('hours_from_kickoff', T.DoubleType(), False),
    T.StructField('weather_lookup_key', T.StringType(), False),
    T.StructField('stadium_id', T.LongType(), False),
    T.StructField('stadium_name', T.StringType(), False),
    T.StructField('city_name', T.StringType(), False),
    T.StructField('country_name', T.StringType(), False),
    T.StructField('latitude', T.DoubleType(), False),
    T.StructField('longitude', T.DoubleType(), False),
    T.StructField('openmeteo_timezone', T.StringType(), False),
    T.StructField('temperature_c', T.DoubleType(), False),
    T.StructField('feels_like_c', T.DoubleType(), False),
    T.StructField('precipitation_mm', T.DoubleType(), False),
    T.StructField('rain_mm', T.DoubleType(), False),
    T.StructField('rain_flag', T.BooleanType(), False),
    T.StructField('temperature_group', T.StringType(), False),
    T.StructField('weather_missing_flag', T.BooleanType(), False),
    T.StructField('tournament_short_name', T.StringType(), False),
    T.StructField('competition_name', T.StringType(), False),
    T.StructField('season_name', T.StringType(), False),
    T.StructField('schema_version', T.StringType(), False),
])

parsed = kafka_strings.withColumn('data', F.from_json('message_value', weather_schema))
malformed_json_count = parsed.filter(F.col('data').isNull()).count()
assert malformed_json_count == 0, f'Malformed Kafka JSON messages: {malformed_json_count}'

weather_from_kafka_raw = parsed.select(
    'topic', 'partition', 'offset', 'kafka_timestamp', 'message_key',
    'data.*',
).withColumn('message_key_match_id', F.col('message_key').cast('long'))

key_mismatch_count = weather_from_kafka_raw.filter(
    F.col('message_key_match_id') != F.col('match_id')
).count()
assert key_mismatch_count == 0, f'Kafka key and JSON match_id mismatch count: {key_mismatch_count}'

weather_from_kafka_raw.printSchema()
weather_from_kafka_raw.orderBy('match_id', 'offset').show(5, truncate=False)

## Deduplizieren und typisieren

Falls der Producer mehrfach ausgefuehrt wurde, bleiben mehrere Kafka-Offsets pro `match_id` im Topic. Fuer die Silver-Tabelle wird pro `match_id` die neueste Kafka-Message verwendet.

In [ ]:
window_by_match = Window.partitionBy('match_id').orderBy(F.col('offset').desc())

weather_from_kafka = (
    weather_from_kafka_raw
    .withColumn('_row_number', F.row_number().over(window_by_match))
    .filter(F.col('_row_number') == 1)
    .drop('_row_number')
    .select(
        F.col('match_id').cast('long'),
        F.to_date('match_date').alias('match_date'),
        F.col('kick_off'),
        F.to_timestamp('kickoff_timestamp').alias('kickoff_timestamp'),
        F.to_timestamp('weather_time').alias('weather_time'),
        F.col('hours_from_kickoff').cast('double'),
        F.col('weather_lookup_key'),
        F.col('stadium_id').cast('long'),
        F.col('stadium_name'),
        F.col('city_name'),
        F.col('country_name'),
        F.col('latitude').cast('double'),
        F.col('longitude').cast('double'),
        F.col('openmeteo_timezone'),
        F.col('temperature_c').cast('double'),
        F.col('feels_like_c').cast('double'),
        F.col('precipitation_mm').cast('double'),
        F.col('rain_mm').cast('double'),
        F.col('rain_flag').cast('boolean'),
        F.col('temperature_group'),
        F.col('weather_missing_flag').cast('boolean'),
        F.col('tournament_short_name'),
        F.col('competition_name'),
        F.col('season_name'),
        F.col('schema_version'),
        F.col('topic').alias('kafka_topic'),
        F.col('partition').alias('kafka_partition'),
        F.col('offset').alias('kafka_offset'),
        F.col('kafka_timestamp'),
    )
    .orderBy('match_id')
)

unique_message_keys = weather_from_kafka_raw.select('match_id').distinct().count()
deduplicated_count = weather_from_kafka.count()
duplicate_count = raw_message_count - deduplicated_count
duplicate_key_messages = raw_message_count - unique_message_keys

assert deduplicated_count == weather_from_kafka.select('match_id').distinct().count()
assert deduplicated_count == expected_match_count(), f'Expected {expected_match_count()} unique matches, got {deduplicated_count}.'
assert weather_from_kafka.filter(F.col('schema_version') != WEATHER_SCHEMA_VERSION).count() == 0

{
    'raw_kafka_messages': raw_message_count,
    'unique_message_keys': unique_message_keys,
    'deduplicated_weather_rows': deduplicated_count,
    'duplicates_removed': duplicate_count,
    'duplicate_key_messages': duplicate_key_messages,
    'deduplication_rule': 'latest offset per match_id',
}


## Ergebnis speichern und pruefen

Das Parquet-Ergebnis ist der stabile Input fuer die nachfolgenden Join-Schritte. Eine kleine Statusdatei dokumentiert die Kafka-Offsets, die Anzahl gelesener Messages und die Deduplizierung.

In [ ]:
project_root = get_project_root()
weather_output_path = project_root / 'data' / 'silver' / 'weather_from_kafka.parquet'
status_output_path = project_root / 'outputs' / 'tables' / 'kafka_weather_read_status.csv'
status_output_path.parent.mkdir(parents=True, exist_ok=True)

weather_from_kafka.write.mode('overwrite').parquet(str(weather_output_path))

written = spark.read.parquet(str(weather_output_path))
written_count = written.count()
assert written_count == deduplicated_count
assert written.select('match_id').distinct().count() == written_count

offset_summary = kafka_strings.agg(
    F.min('offset').alias('min_offset'),
    F.max('offset').alias('max_offset'),
).collect()[0].asDict()

status = {
    'topic': KAFKA_TOPIC,
    'bootstrap_servers': KAFKA_BOOTSTRAP_SERVERS,
    'raw_kafka_messages': raw_message_count,
    'unique_message_keys': unique_message_keys,
    'deduplicated_weather_rows': deduplicated_count,
    'duplicates_removed': duplicate_count,
    'duplicate_key_messages': duplicate_key_messages,
    'deduplication_rule': 'latest offset per match_id',
    'malformed_json_messages': malformed_json_count,
    'key_mismatch_messages': key_mismatch_count,
    'min_offset': offset_summary['min_offset'],
    'max_offset': offset_summary['max_offset'],
    'output_path': 'data/silver/weather_from_kafka.parquet',
}

import pandas as pd
pd.DataFrame([status]).to_csv(status_output_path, index=False)

written.orderBy('match_id').show(5, truncate=False)
status


## Ergebnis

BD-15 ist erfuellt, wenn die Kafka-Messages mit Spark gelesen, das JSON-Schema erfolgreich angewendet, Duplikate ueber `match_id` entfernt und `data/silver/weather_from_kafka.parquet` geschrieben wurde.

In [ ]:
spark.stop()